# StyleFinder — Task 2 Results Visualization

Run this notebook **after** `build_index.py` has been executed. It loads all indexes from disk and generates visualizations.

In [ ]:
%cd /content/stylefinder
!pip install -q umap-learn plotly matplotlib

In [ ]:
import sys, os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import faiss
import sqlite3
import umap
import plotly.express as px
from sentence_transformers import SentenceTransformer
from IPython.display import display, HTML

sys.path.insert(0, ".")
from src.config import *
from src.search import StyleFinderEngine

In [ ]:
engine = StyleFinderEngine()

# Also load raw data for visualizations
from datasets import load_dataset
product_dataset = load_dataset("ashraq/fashion-product-images-small", split="train")
product_df = product_dataset.to_pandas()
product_df = product_df.dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)
style_df = load_dataset("neuralwork/fashion-style-instruct", split="train").to_pandas()

# Load embeddings from FAISS (reconstruct)
style_faiss = faiss.read_index(STYLE_FAISS)
product_faiss = faiss.read_index(PRODUCT_FAISS)
n_style = style_faiss.ntotal
n_product = product_faiss.ntotal
dim = 384

style_embeddings = np.vstack([style_faiss.reconstruct(i) for i in range(n_style)]).astype('float32')
product_embeddings = np.vstack([product_faiss.reconstruct(i) for i in range(n_product)]).astype('float32')
print(f"Style embeddings: {style_embeddings.shape}")
print(f"Product embeddings: {product_embeddings.shape}")

## 1. Index Summary

In [ ]:
# File sizes
files = []
for fname in sorted(os.listdir(INDEX_DIR)):
    size = os.path.getsize(os.path.join(INDEX_DIR, fname)) / 1024 / 1024
    files.append({"File": fname, "Size (MB)": round(size, 2)})

df_files = pd.DataFrame(files)
total = df_files["Size (MB)"].sum()
df_files.loc[len(df_files)] = {"File": "TOTAL", "Size (MB)": round(total, 2)}

display(HTML("<h4>Persisted Index Files</h4>"))
display(df_files.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

# Summary stats
summary = {
    "Index": ["FAISS (style)", "FAISS (product)", "SQLite (products)", "SQLite (styles)"],
    "Items": [n_style, n_product, len(product_df), len(style_df)],
    "Type": ["384-dim vectors", "384-dim vectors", "Metadata rows", "Metadata rows"],
    "Search": ["Semantic", "Semantic", "SQL filter", "SQL filter"]
}
display(HTML("<h4>Index Summary</h4>"))
display(pd.DataFrame(summary).style.hide(axis='index'))

## 2. Product Catalog Overview

In [ ]:
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 5))
fig.suptitle("Product Catalog Overview (42,587 products)", fontsize=16, fontweight='bold', y=1.02)
gs = gridspec.GridSpec(1, 3, width_ratios=[0.8, 1.2, 1.0], wspace=0.3)

# 1. GENDER — Donut chart
ax1 = fig.add_subplot(gs[0])
gender_counts = product_df['gender'].value_counts()
gender_counts = gender_counts[gender_counts.index.isin(['Men', 'Women', 'Unisex'])]
colors_gender = ['#1E2761', '#F96167', '#43A047']
wedges, texts, autotexts = ax1.pie(
    gender_counts.values, labels=None, autopct='%1.0f%%',
    colors=colors_gender, pctdistance=0.75, startangle=90,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2.5)
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
    t.set_color('white')
ax1.legend(
    [f"{g} ({v:,})" for g, v in zip(gender_counts.index, gender_counts.values)],
    loc='center', fontsize=10, frameon=False,
    bbox_to_anchor=(0.5, -0.05)
)
ax1.set_title("By Gender", fontweight='bold', fontsize=13, pad=15)

# 2. ARTICLE TYPES — Horizontal bar (keep)
ax2 = fig.add_subplot(gs[1])
top_types = product_df['articleType'].value_counts().head(8)
colors_types = ['#1E2761', '#F96167', '#43A047', '#5267AC', '#6A80C4', '#8A9DD6', '#A8B8E0', '#C5D0EC']
bars2 = ax2.barh(top_types.index, top_types.values, color=blues, height=0.6)
ax2.set_title("Top 8 Article Types", fontweight='bold', fontsize=13)
ax2.invert_yaxis()
ax2.set_xlim(0, top_types.max() * 1.15)
for bar, val in zip(bars2, top_types.values):
    ax2.text(val + 50, bar.get_y() + bar.get_height()/2, f"{val:,}", va='center', fontsize=9, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# 3. USAGE — Treemap
ax3 = fig.add_subplot(gs[2])
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 10)
ax3.set_aspect('equal')
ax3.axis('off')
ax3.set_title("By Usage", fontweight='bold', fontsize=13)

usage_counts = product_df['usage'].dropna().value_counts()
total = usage_counts.sum()
colors_usage = ['#1E2761', '#F96167', '#43A047', '#FF9800', '#9C27B0', '#BDBDBD']

# Main block: Casual (78%)
casual_pct = usage_counts.iloc[0] / total
rect = FancyBboxPatch((0.2, 2.0), 9.5, 7.8, boxstyle="round,pad=0.1",
                       facecolor=colors_usage[0], edgecolor='white', linewidth=2)
ax3.add_patch(rect)
ax3.text(5.0, 6.5, "Casual", fontsize=18, fontweight='bold', color='white', ha='center', va='center')
ax3.text(5.0, 5.0, f"{usage_counts.iloc[0]:,} ({casual_pct:.0%})", fontsize=13, color='#CADCFC', ha='center', va='center')

# Smaller blocks for the rest
others = usage_counts.iloc[1:5]
x_positions = [0.2, 2.6, 5.0, 7.4]
for i, (name, val) in enumerate(others.items()):
    pct = val / total
    rect = FancyBboxPatch((x_positions[i], 0.1), 2.2, 1.6, boxstyle="round,pad=0.05",
                           facecolor=colors_usage[i+1], edgecolor='white', linewidth=1.5)
    ax3.add_patch(rect)
    ax3.text(x_positions[i] + 1.1, 1.1, name, fontsize=8, fontweight='bold', color='white', ha='center', va='center')
    ax3.text(x_positions[i] + 1.1, 0.5, f"{pct:.0%}", fontsize=9, color='white', ha='center', va='center')

plt.tight_layout()
plt.savefig("product_overview.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Style Dataset Overview

In [ ]:
occasions = style_df['context'].value_counts().head(12)

# Clean labels: remove "I'm going to a" prefix
clean_labels = [o.replace("I'm going to a ", "").replace("I'm going to a", "").strip().rstrip('.').title()
                for o in occasions.index]

fig, ax = plt.subplots(figsize=(10, 5))
colors_occ = ['#F96167' if i < 4 else '#FF9E9E' if i < 8 else '#FFD0D0' for i in range(len(occasions))]
bars = ax.barh(clean_labels, occasions.values, color=colors_occ, height=0.65)
ax.set_title("Style Dataset: Top 12 Occasions (3,193 entries)", fontweight='bold', fontsize=14)
ax.invert_yaxis()
ax.set_xlim(80, occasions.max() * 1.08)  # start at 80 to show differences
for bar, val in zip(bars, occasions.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=10, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel("Number of entries", fontsize=11)

# Add note about even distribution
ax.text(0.95, 0.02, "Note: nearly uniform distribution (88-96 per occasion)",
        transform=ax.transAxes, fontsize=9, color='gray', ha='right', style='italic')

plt.tight_layout()
plt.show()



## 4. UMAP: Product Embeddings by Category

In [ ]:
np.random.seed(42)
sample_n = 3000
sample_idx = np.random.choice(n_product, sample_n, replace=False)
sample_embs = product_embeddings[sample_idx]

reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, n_components=2, random_state=42)
umap_2d = reducer.fit_transform(sample_embs)

categories = [product_df.iloc[i]['masterCategory'] for i in sample_idx]
cat_map = {'Apparel': '#E53935', 'Footwear': '#1E88E5', 'Accessories': '#43A047',
           'Personal Care': '#8E24AA', 'Free Items': '#FF9800'}

fig, ax = plt.subplots(figsize=(12, 8))

# Plot smaller categories first, then larger ones on top
for cat in ['Free Items', 'Personal Care', 'Accessories', 'Footwear', 'Apparel']:
    mask = [c == cat for c in categories]
    if any(mask):
        pts = umap_2d[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=cat_map[cat], label=cat, s=8, alpha=0.5)

ax.legend(fontsize=11, markerscale=4, framealpha=0.9)
ax.set_title("UMAP: Product Embeddings by Category (n=3,000)", fontsize=14, fontweight='bold')
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.grid(True, alpha=0.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("umap_products.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. UMAP: Style Recommendations

In [ ]:
reducer_s = umap.UMAP(n_neighbors=10, min_dist=0.1, n_components=2, random_state=42)
style_2d = reducer_s.fit_transform(style_embeddings)

# Color by occasion type (group similar ones)
def occasion_group(ctx):
    ctx = str(ctx).lower()
    if 'wedding' in ctx: return 'Wedding'
    if 'interview' in ctx or 'business' in ctx or 'work' in ctx or 'office' in ctx: return 'Professional'
    if 'date' in ctx: return 'Date'
    if 'party' in ctx or 'club' in ctx or 'gala' in ctx: return 'Party/Night'
    if 'hiking' in ctx or 'camping' in ctx or 'national' in ctx or 'retreat' in ctx: return 'Outdoor'
    if 'beach' in ctx or 'tropical' in ctx or 'cruise' in ctx: return 'Beach/Travel'
    return 'Other'

occ_groups = [occasion_group(style_df.iloc[i]['context']) for i in range(len(style_df))]
occ_colors = {
    'Wedding': '#E53935',      # red
    'Professional': '#1E2761', # dark navy
    'Date': '#F06292',         # pink
    'Party/Night': '#8E24AA',  # purple
    'Outdoor': '#43A047',      # green
    'Beach/Travel': '#FF9800', # orange
    'Other': '#BDBDBD'         # gray
}

fig, ax = plt.subplots(figsize=(12, 8))
for occ, color in occ_colors.items():
    mask = [o == occ for o in occ_groups]
    if any(mask):
        pts = style_2d[mask]
        ax.scatter(pts[:, 0], pts[:, 1], c=color, label=occ, s=12, alpha=0.5)

ax.legend(fontsize=10, markerscale=3)
ax.set_title("UMAP: Style Recommendations by Occasion Type", fontsize=14, fontweight='bold')
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("umap_styles.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. StyleFinder Demo with Product Images

In [ ]:
def stylefinder_visual(query, n_styles=2, n_products=5):
    print(f"\nQuery: {query}")
    print("=" * 70)

    # Layer 1
    style_results = engine.faiss_search(query, layer="style", n=n_styles)
    print("\n--- LAYER 1: Style Recommendations ---")
    for r in style_results:
        # Get the original style entry
        sid = int(r['doc_id'])
        row = style_df.iloc[sid]
        print(f"\n  Score: {r['score']:.4f}")
        print(f"  Profile: {row['input'][:100]}...")
        print(f"  Occasion: {row['context']}")
        # Show first outfit only
        comp = row['completion']
        end = comp.find("Outfit 2")
        if end == -1: end = comp.find("2. Outfit")
        if end == -1: end = comp.find("Outfit Combination 2")
        if end == -1: end = 400
        print(f"  Outfit: {comp[:end].strip()[:300]}")

    # Layer 2
    best_text = engine.style_corpus[style_results[0]["doc_id"]][:300]
    product_query = query + " " + best_text
    product_results = engine.faiss_search(product_query, layer="product", n=n_products)

    print(f"\n--- LAYER 2: Matching Products ---")
    fig, axes = plt.subplots(1, n_products, figsize=(3.5 * n_products, 4))
    if n_products == 1: axes = [axes]

    for i, r in enumerate(product_results):
        pid = int(r['doc_id'])
        row = product_df.iloc[pid]
        print(f"  {i+1}. (score: {r['score']:.4f}) {row['productDisplayName']} | {row['baseColour']} {row['articleType']}")

        img = product_dataset[int(product_df.iloc[pid].name)]['image']
        axes[i].imshow(img)
        axes[i].set_title(f"{row['productDisplayName'][:28]}\n{row['baseColour']} | {row['articleType']}\nscore: {r['score']:.3f}",
                         fontsize=8, fontweight='bold')
        axes[i].axis('off')

    plt.suptitle(f"Products for: {query}", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
stylefinder_visual("I have an hourglass figure and need something elegant for a summer wedding")

In [ ]:
stylefinder_visual("plus-size woman preparing for a job interview")

In [ ]:
stylefinder_visual("comfortable hiking outfit for cool autumn weather")

In [ ]:
stylefinder_visual("non-binary person minimalist outfit for an art gallery opening")

## 7. SQL + FAISS Hybrid Search Demo

In [ ]:
def filtered_visual(query, gender=None, article_type=None, colour=None, n=5):
    print(f"Query: {query}")
    filters = []
    if gender: filters.append(f"gender={gender}")
    if article_type: filters.append(f"type={article_type}")
    if colour: filters.append(f"colour={colour}")
    print(f"SQL Filters: {', '.join(filters) if filters else 'none'}")
    print("-" * 50)

    results = engine.filtered_search(query, gender=gender, article_type=article_type, colour=colour, n=n)
    if not results:
        print("  No matching products.")
        return

    print(f"  Found {len(results)} results after SQL filtering + FAISS ranking\n")

    fig, axes = plt.subplots(1, len(results), figsize=(3.5 * len(results), 4))
    if len(results) == 1: axes = [axes]

    for i, r in enumerate(results):
        pid = int(r['doc_id'])
        row = product_df.iloc[pid]
        img = product_dataset[int(product_df.iloc[pid].name)]['image']
        axes[i].imshow(img)
        axes[i].set_title(f"{r['name'][:28]}\n{r['colour']} | {r['type']}\nscore: {r['score']:.3f}",
                         fontsize=8, fontweight='bold')
        axes[i].axis('off')

    plt.suptitle(f"SQL+FAISS: {query} ({', '.join(filters)})", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

filtered_visual("elegant evening outfit", gender="Women", article_type="Dresses", colour="Black")

In [ ]:
filtered_visual("casual weekend shirt", gender="Men", article_type="Shirts", colour="Blue")

In [ ]:
filtered_visual("sporty running shoes", gender="Women", article_type="Sports Shoes")

## 8. Evaluation Results

In [ ]:
# Load evaluation set
with open("evaluation_set.json", "r") as f:
    eval_set = json.load(f)

# Run evaluation
rows = []
for entry in eval_set:
    results = engine.faiss_search(entry["query"], layer=entry["layer"], n=5)
    retrieved = [r["doc_id"] for r in results]
    expected = entry["expected_doc_ids"]
    top_score = results[0]["score"] if results else 0

    if expected:
        hits = len([d for d in retrieved if d in expected])
        recall = hits / len(expected)
        qtype = "Realistic"
    else:
        hits = 0
        recall = None
        qtype = "Adversarial"

    rows.append({
        "ID": entry["query_id"],
        "Query": entry["query"][:45] + "...",
        "Layer": entry["layer"],
        "Type": qtype,
        "Expected": len(expected),
        "Hits@5": hits,
        "Recall": f"{recall:.2f}" if recall is not None else "N/A",
        "Top Score": f"{top_score:.4f}"
    })

df_eval = pd.DataFrame(rows)

def color_recall(val):
    if val == "N/A": return "background-color: #FFE0B2"
    v = float(val)
    if v >= 0.8: return "background-color: #C8E6C9"
    if v >= 0.4: return "background-color: #FFF9C4"
    return "background-color: #FFCDD2"

display(HTML("<h4>Evaluation Results: 15 Queries</h4>"))
display(df_eval.style.applymap(color_recall, subset=["Recall"]).hide(axis='index'))

# Summary stats
realistic = df_eval[df_eval["Type"] == "Realistic"]
with_hits = realistic[realistic["Hits@5"].astype(int) > 0]
print(f"\nRealistic queries with at least 1 hit: {len(with_hits)}/{len(realistic)}")
total_hits = realistic["Hits@5"].astype(int).sum()
total_expected = realistic["Expected"].astype(int).sum()
print(f"Overall recall: {total_hits}/{total_expected} = {total_hits/total_expected:.2f}")

In [ ]:
# Load evaluation results
with open("evaluation_set.json", "r") as f:
    eval_set = json.load(f)

# Run evaluation
rows = []
for entry in eval_set:
    results = engine.faiss_search(entry["query"], layer=entry["layer"], n=5)
    retrieved = [r["doc_id"] for r in results]
    expected = entry["expected_doc_ids"]
    top_score = results[0]["score"] if results else 0

    if expected:
        hits = len([d for d in retrieved if d in expected])
        recall = hits / len(expected)
        qtype = "Realistic"
    else:
        hits = 0
        recall = None
        qtype = "Adversarial"

    rows.append({
        "id": entry["query_id"],
        "query": entry["query"],
        "type": qtype,
        "layer": entry["layer"],
        "recall": recall,
        "top_score": top_score,
        "hits": hits,
        "expected": len(expected)
    })

# ---- Chart 1: Recall bars (realistic only) ----
realistic = [r for r in rows if r["type"] == "Realistic"]

fig, ax = plt.subplots(figsize=(14, 5))

# Short labels
short_labels = {
    "Q01": "Wedding dress\n(style)",
    "Q02": "Plus-size\ninterview (style)",
    "Q03": "Tall man\ndate (style)",
    "Q04": "Non-binary\ngallery (style)",
    "Q05": "Pear-shaped\nbrunch (style)",
    "Q06": "Hiking outfit\n(style)",
    "Q07": "Hourglass\ngown (style)",
    "Q08": "Petite festival\n(style)",
    "Q09": "Black dress\n(product)",
    "Q10": "Blue shirt\n(product)",
    "Q11": "White sneakers\n(product)",
    "Q12": "Formal shoes\n(product)",
}

x_labels = [short_labels.get(r["id"], r["id"]) for r in realistic]
recalls = [r["recall"] for r in realistic]
colors = ['#43A047' if v >= 0.8 else '#FF9800' if v >= 0.4 else '#E53935' for v in recalls]

bars = ax.bar(range(len(realistic)), recalls, color=colors, width=0.6, edgecolor='white', linewidth=1)

# Value labels
for i, (bar, val, r) in enumerate(zip(bars, recalls, realistic)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.2f}", ha='center', fontsize=10, fontweight='bold')
    # Show hits/expected below the bar
    ax.text(bar.get_x() + bar.get_width()/2, -0.08,
            f"{r['hits']}/{r['expected']}", ha='center', fontsize=8, color='gray')

ax.set_xticks(range(len(realistic)))
ax.set_xticklabels(x_labels, fontsize=8, ha='center')
ax.set_ylim(-0.12, 1.15)
ax.set_ylabel("Recall@5", fontsize=12, fontweight='bold')
ax.set_title("Evaluation: Recall per Query (12 Realistic Queries)", fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Separator line between style and product queries
ax.axvline(x=7.5, color='gray', linestyle=':', alpha=0.5)
ax.text(3.5, 1.1, "Style Queries", ha='center', fontsize=10, color='#1E2761', fontweight='bold')
ax.text(9.5, 1.1, "Product Queries", ha='center', fontsize=10, color='#1E2761', fontweight='bold')

legend_patches = [
    mpatches.Patch(color='#43A047', label='High (>= 0.8)'),
    mpatches.Patch(color='#FF9800', label='Medium (>= 0.4)'),
    mpatches.Patch(color='#E53935', label='Low (< 0.4)')
]
ax.legend(handles=legend_patches, loc='center right', fontsize=9)

plt.tight_layout()
plt.savefig("recall_per_query.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Interactive UMAP (Plotly)

In [ ]:
# Style embeddings with hover
labels = [f"{style_df.iloc[i]['input'][:50]}... | {style_df.iloc[i]['context']}" for i in range(len(style_df))]
groups = [occasion_group(style_df.iloc[i]['context']) for i in range(len(style_df))]

fig = px.scatter(x=style_2d[:, 0], y=style_2d[:, 1], color=groups, hover_name=labels,
                 title="Interactive UMAP: Style Recommendations (hover for details)",
                 color_discrete_map=occ_colors, opacity=0.6)
fig.update_traces(marker_size=5)
fig.update_layout(height=600)
fig.show()

In [ ]:
engine.close()
